# T8-bonus · Policy as code

## Goal

Move the DLP policy and managed-environment sharing limits fully into
Terraform, and add the CI gate that fails a PR if a guard test is deleted
from `evals/golden_cases.json` without a matching removal in the policy
itself.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../infra/terraform/platform/dlp.tf").exists()


## Concept

`24` proved each guard has a test. This notebook is the meta-guard: a CI
check that catches someone quietly deleting a `governance`-tagged golden
case to make a red suite go green, rather than fixing the underlying
policy. That's the actual failure mode worth defending against — not
"someone forgot a test," but "someone removed an inconvenient one."


## Build


In [ ]:
governance_case_count_floor = 3  # gov-01, gov-02, gov-03 — bump this deliberately if you add more, never lower it silently
import json
cases = json.load(open("../evals/golden_cases.json"))
governance_cases = [c for c in cases if "governance" in c.get("tags", [])]
assert len(governance_cases) >= governance_case_count_floor, (
    f"governance case count dropped below the floor ({len(governance_cases)} < {governance_case_count_floor}) — "
    f"if this is deliberate, raise the floor in this cell in the same PR, don't just let the count drop"
)
print(f"{len(governance_cases)} governance cases present, floor is {governance_case_count_floor}")


In [ ]:
ci_check = '''
name: guard-floor-check
on: [pull_request]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: python -c "
          import json;
          cases = json.load(open('evals/golden_cases.json'));
          gov = [c for c in cases if 'governance' in c.get('tags', [])];
          assert len(gov) >= 3, f'governance case count dropped: {len(gov)}'
          "
'''
from pathlib import Path
Path("../infra/pipelines/guard-floor-check.yml").write_text(ci_check)
print("guard-floor-check.yml written — this is the CI gate that catches a silently-deleted guard test")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
# Simulate the failure this gate exists to catch
import json
cases = json.load(open("../evals/golden_cases.json"))
tampered = [c for c in cases if not (c["id"] == "gov-03-content-safety")]
tampered_gov = [c for c in tampered if "governance" in c.get("tags", [])]
try:
    assert len(tampered_gov) >= governance_case_count_floor
    print("gate did not catch the tampering — floor is set too low")
except AssertionError:
    print("gate correctly caught the tampering: removing gov-03 without raising the floor fails CI")


## Cost


In [ ]:
print("No agent build/publish — this notebook is CI/policy tooling only.")


## Teardown


In [ ]:
print("No teardown — the guard-floor gate is now a standing CI check.")
